In [1]:
import json

In [2]:
with open("all_results/scores.json") as f:
    data = json.load(f)


In [3]:
len(data)

13

In [4]:
import numpy as np
from statsmodels.stats.contingency_tables import mcnemar
import pandas as pd

In [5]:
ref_key = list(data.keys())[5]

In [6]:
for key in data:
    total = []
    for field in data[key]:
        if field != "experimental_factors_20":
            total.extend(data[key][field])
    data[key]["total"] = total

In [7]:
#data[ref_key]

In [8]:
ref_key

"[('answer_in_quotes', True), ('context_shortener', 'keybert-literal'), ('dataset', 'arxpr2s_25'), ('ff_model', 'llama3.1I-8b-q4'), ('keyphrase_min', 1), ('keyphrase_range_diff', 1), ('listed_output', False), ('log', True), ('maxsum_factor', 1.0), ('mmr_param', 1.0), ('n_keywords', 5), ('openai_ff_max_tokens', 1000), ('outlines_ff_max_tokens', 100), ('rag_chunk_overlap', 128), ('rag_chunk_size', 2048), ('rag_llm', 'llama3.1'), ('reduce_chunk_overlap', 100), ('reduce_chunk_size', 300), ('reduce_max_tokens', 50), ('reduce_temperature', 0.4), ('remove_fields', 'default'), ('retriever_type', 'simple'), ('sampler', 'beam'), ('sampler_beams', 2), ('sampler_temp', 0.4), ('sampler_top_k', 100), ('sampler_top_p', 1), ('similarity_k', 4)]"

In [9]:
def pval(x1, x2, verbose = True): # x2 is ref
    if not len(x1)==len(x2):
        return

    assert len(x1)==len(x2), (len(x1),len(x2))

    a,b,c,d = 0,0,0,0
    for i in range(len(x1)):
        a += x1[i] and x2[i]
        b += x1[i] and not x2[i]
        c += not x1[i] and x2[i]
        d += not x1[i] and not x2[i]

    mcnemar_matrix = [[a, b], [c, d]]

    # all of them:
    #print(mcnemar(mcnemar_matrix, exact=True))
    #print(mcnemar(mcnemar_matrix, exact=False))
    #print(mcnemar(mcnemar_matrix, exact=False, correction=False))

    # most correct(?)
    result = mcnemar(mcnemar_matrix, exact=(b+c<25), correction=min(a,b,c,d)>25)
    if verbose:

        print(field)
        print("reference better" if c>b else "reference WORSE")
        print(c, b)
        print(result)
        print("")
    return result.pvalue if c>b else np.nan#1/result.pvalue
    

In [10]:


for key in data:
    print("")
    print(key)
    for field in data[key]:
        ref_array = data[ref_key][field]
        array = data[key][field]
        pval(array, ref_array)



[('answer_in_quotes', True), ('context_shortener', 'full_paper'), ('dataset', 'arxpr2s_25'), ('ff_model', '4om'), ('keyphrase_min', 1), ('keyphrase_range_diff', 1), ('listed_output', False), ('log', True), ('maxsum_factor', 1.0), ('mmr_param', 1.0), ('n_keywords', 5), ('openai_ff_max_tokens', 1000), ('outlines_ff_max_tokens', 100), ('rag_chunk_overlap', 128), ('rag_chunk_size', 2048), ('rag_llm', 'llama3.1'), ('reduce_chunk_overlap', 400), ('reduce_chunk_size', 20000), ('reduce_max_tokens', 50), ('reduce_temperature', 0.4), ('remove_fields', 'default'), ('retriever_type', 'simple'), ('sampler', 'beam'), ('sampler_beams', 2), ('sampler_temp', 0.4), ('sampler_top_k', 100), ('sampler_top_p', 1), ('similarity_k', 1)]
hardware_4
reference WORSE
3.0 48.0
pvalue      2.95234941290252e-10
statistic   39.705882352941174

organism_part_5
reference WORSE
6.0 22.0
pvalue      0.002496908915141551
statistic   9.142857142857142

experimental_designs_10
reference WORSE
1.0 2.0
pvalue      1.0
statis

In [11]:
for key in data:
    print(key)
    print(len(data[key]["total"]))

[('answer_in_quotes', True), ('context_shortener', 'full_paper'), ('dataset', 'arxpr2s_25'), ('ff_model', '4om'), ('keyphrase_min', 1), ('keyphrase_range_diff', 1), ('listed_output', False), ('log', True), ('maxsum_factor', 1.0), ('mmr_param', 1.0), ('n_keywords', 5), ('openai_ff_max_tokens', 1000), ('outlines_ff_max_tokens', 100), ('rag_chunk_overlap', 128), ('rag_chunk_size', 2048), ('rag_llm', 'llama3.1'), ('reduce_chunk_overlap', 400), ('reduce_chunk_size', 20000), ('reduce_max_tokens', 50), ('reduce_temperature', 0.4), ('remove_fields', 'default'), ('retriever_type', 'simple'), ('sampler', 'beam'), ('sampler_beams', 2), ('sampler_temp', 0.4), ('sampler_top_k', 100), ('sampler_top_p', 1), ('similarity_k', 1)]
500
[('answer_in_quotes', True), ('context_shortener', 'keyword-literal'), ('dataset', 'arxpr2s_25'), ('ff_model', 'keybert'), ('keyphrase_min', 1), ('keyphrase_range_diff', 1), ('listed_output', False), ('log', True), ('maxsum_factor', 1.0), ('mmr_param', 1.0), ('n_keywords',

In [14]:
shortnames = [
    "fullpaper_gpt",
    "keyword_keyword",
    "keyword_gpt",
    "keyword_llama",
    "keybert_gpt",
    "keybert_llama",
    "rag_gpt_17",
    "rag_gpt_10",
    "rag_llama",
    "keybert_keyword",
    "keyword_mistral",
    "keylist_gpt",
    "keydescript_gpt",
]

In [15]:
dfs = {}
for ref_key in data:
    values = []
    for key in data:
        values.append([])
        for field in data[key]:
            values[-1].append(pval(data[key][field], data[ref_key][field], False))
    df = pd.DataFrame(data=values, columns= data[key].keys(), index=shortnames)
    dfs[ref_key] = df

In [16]:
def p(i):
    k = list(dfs.keys())[i]
    print(shortnames[i])
    print(k)
    return dfs[k]

In [17]:
p(0)

fullpaper_gpt
[('answer_in_quotes', True), ('context_shortener', 'full_paper'), ('dataset', 'arxpr2s_25'), ('ff_model', '4om'), ('keyphrase_min', 1), ('keyphrase_range_diff', 1), ('listed_output', False), ('log', True), ('maxsum_factor', 1.0), ('mmr_param', 1.0), ('n_keywords', 5), ('openai_ff_max_tokens', 1000), ('outlines_ff_max_tokens', 100), ('rag_chunk_overlap', 128), ('rag_chunk_size', 2048), ('rag_llm', 'llama3.1'), ('reduce_chunk_overlap', 400), ('reduce_chunk_size', 20000), ('reduce_max_tokens', 50), ('reduce_temperature', 0.4), ('remove_fields', 'default'), ('retriever_type', 'simple'), ('sampler', 'beam'), ('sampler_beams', 2), ('sampler_temp', 0.4), ('sampler_top_k', 100), ('sampler_top_p', 1), ('similarity_k', 1)]


,hardware_4,organism_part_5,experimental_designs_10,assay_by_molecule_14,study_type_18,experimental_factors_20,total
fullpaper_gpt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keyword_keyword,9.410951e-08,3.613113e-05,NaN,5.878172e-02,0.001463,0.503445,8.261718e-03
keyword_gpt,NaN,NaN,NaN,1.000000e+00,NaN,NaN,NaN
keyword_llama,8.105017e-10,1.586133e-02,1.0,5.925254e-06,NaN,0.034690,4.910431e-09
keybert_gpt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keybert_llama,2.952349e-10,2.496909e-03,1.0,9.758408e-06,NaN,0.052479,1.505272e-06
rag_gpt_17,NaN,NaN,NaN,2.209050e-05,NaN,NaN,NaN
rag_gpt_10,2.314093e-02,2.568393e-01,NaN,1.606146e-07,0.303484,NaN,4.870841e-04
rag_llama,1.135214e-11,2.109234e-08,0.5,4.663011e-11,0.140369,0.016901,2.942551e-25
keybert_keyword,2.559625e-12,3.957025e-05,NaN,1.305700e-01,0.006611,0.189247,1.456783e-05


In [18]:
p(1)

keyword_keyword
[('answer_in_quotes', True), ('context_shortener', 'keyword-literal'), ('dataset', 'arxpr2s_25'), ('ff_model', 'keybert'), ('keyphrase_min', 1), ('keyphrase_range_diff', 1), ('listed_output', False), ('log', True), ('maxsum_factor', 1.0), ('mmr_param', 1.0), ('n_keywords', 5), ('openai_ff_max_tokens', 1000), ('outlines_ff_max_tokens', 100), ('rag_chunk_overlap', 128), ('rag_chunk_size', 2048), ('rag_llm', 'llama3.1'), ('reduce_chunk_overlap', 100), ('reduce_chunk_size', 500), ('reduce_max_tokens', 50), ('reduce_temperature', 0.4), ('remove_fields', 'default'), ('retriever_type', 'simple'), ('sampler', 'beam'), ('sampler_beams', 2), ('sampler_temp', 0.4), ('sampler_top_k', 100), ('sampler_top_p', 1), ('similarity_k', 1)]


,hardware_4,organism_part_5,experimental_designs_10,assay_by_molecule_14,study_type_18,experimental_factors_20,total
fullpaper_gpt,NaN,NaN,2.218341e-12,NaN,NaN,NaN,NaN
keyword_keyword,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keyword_gpt,NaN,NaN,6.249041e-05,NaN,NaN,NaN,NaN
keyword_llama,0.803619,NaN,5.351800e-13,2.346726e-03,NaN,0.143463,2.118617e-03
keybert_gpt,NaN,NaN,9.448129e-05,NaN,NaN,NaN,NaN
keybert_llama,0.423950,NaN,5.351800e-13,4.620684e-03,NaN,0.237885,7.857855e-02
rag_gpt_17,NaN,NaN,4.138729e-08,1.332833e-02,NaN,NaN,NaN
rag_gpt_10,NaN,NaN,2.906095e-08,3.857468e-04,NaN,NaN,7.710841e-01
rag_llama,0.065430,0.009023,1.205298e-13,6.776262e-08,NaN,0.096252,2.479042e-15
keybert_keyword,0.021484,0.790527,8.554459e-04,NaN,NaN,0.581055,3.588757e-02


In [19]:
p(2)

keyword_gpt
[('answer_in_quotes', True), ('context_shortener', 'keyword-literal'), ('dataset', 'arxpr2s_25'), ('ff_model', '4om'), ('keyphrase_min', 1), ('keyphrase_range_diff', 1), ('listed_output', False), ('log', True), ('maxsum_factor', 1.0), ('mmr_param', 1.0), ('n_keywords', 5), ('openai_ff_max_tokens', 1000), ('outlines_ff_max_tokens', 100), ('rag_chunk_overlap', 128), ('rag_chunk_size', 2048), ('rag_llm', 'llama3.1'), ('reduce_chunk_overlap', 100), ('reduce_chunk_size', 500), ('reduce_max_tokens', 50), ('reduce_temperature', 0.4), ('remove_fields', 'default'), ('retriever_type', 'simple'), ('sampler', 'beam'), ('sampler_beams', 2), ('sampler_temp', 0.4), ('sampler_top_k', 100), ('sampler_top_p', 1), ('similarity_k', 10)]


,hardware_4,organism_part_5,experimental_designs_10,assay_by_molecule_14,study_type_18,experimental_factors_20,total
fullpaper_gpt,4.343510e-03,5.034447e-01,7.430984e-07,NaN,3.713338e-06,0.332306,1.248369e-10
keyword_keyword,1.825753e-11,2.034555e-07,NaN,3.515625e-02,1.480502e-12,0.093140,1.839730e-16
keyword_gpt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keyword_llama,1.164432e-13,2.607296e-04,1.903182e-07,9.546920e-06,8.235705e-06,0.001702,1.527597e-30
keybert_gpt,1.181793e-02,1.000000e+00,NaN,NaN,NaN,0.266846,7.463179e-01
keybert_llama,4.215201e-14,8.769942e-05,1.903182e-07,1.556804e-05,1.943659e-01,0.002700,2.913025e-24
rag_gpt_17,3.468966e-02,6.476059e-01,9.322376e-03,9.584006e-06,1.832074e-02,NaN,3.940392e-08
rag_gpt_10,3.061253e-06,4.986020e-02,2.699796e-03,1.137273e-07,5.229936e-08,NaN,8.639963e-21
rag_llama,9.189255e-15,5.919399e-10,4.320463e-08,1.781872e-10,1.343993e-08,0.001069,3.264446e-45
keybert_keyword,2.067066e-15,4.458707e-07,NaN,1.670685e-01,1.414994e-10,0.011818,3.264418e-21


In [20]:
p(3)

keyword_llama
[('answer_in_quotes', True), ('context_shortener', 'keyword-literal'), ('dataset', 'arxpr2s_25'), ('ff_model', 'llama3.1I-8b-q4'), ('keyphrase_min', 1), ('keyphrase_range_diff', 1), ('listed_output', False), ('log', True), ('maxsum_factor', 1.0), ('mmr_param', 1.0), ('n_keywords', 5), ('openai_ff_max_tokens', 1000), ('outlines_ff_max_tokens', 100), ('rag_chunk_overlap', 128), ('rag_chunk_size', 2048), ('rag_llm', 'llama3.1'), ('reduce_chunk_overlap', 100), ('reduce_chunk_size', 300), ('reduce_max_tokens', 50), ('reduce_temperature', 0.4), ('remove_fields', 'default'), ('retriever_type', 'simple'), ('sampler', 'beam'), ('sampler_beams', 2), ('sampler_temp', 0.4), ('sampler_top_k', 100), ('sampler_top_p', 1), ('similarity_k', 4)]


,hardware_4,organism_part_5,experimental_designs_10,assay_by_molecule_14,study_type_18,experimental_factors_20,total
fullpaper_gpt,NaN,NaN,NaN,NaN,0.257899,NaN,NaN
keyword_keyword,NaN,0.208668,NaN,NaN,0.000015,NaN,NaN
keyword_gpt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keyword_llama,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keybert_gpt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keybert_llama,0.68750,NaN,NaN,NaN,NaN,NaN,NaN
rag_gpt_17,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rag_gpt_10,NaN,NaN,NaN,0.751830,0.043308,NaN,NaN
rag_llama,0.12500,0.000246,1.0,0.006656,0.009375,1.0,6.582650e-08
keybert_keyword,0.03125,0.128190,NaN,NaN,0.000504,NaN,NaN


In [21]:
p(4)

keybert_gpt
[('answer_in_quotes', True), ('context_shortener', 'keybert-literal'), ('dataset', 'arxpr2s_25'), ('ff_model', '4om'), ('keyphrase_min', 1), ('keyphrase_range_diff', 1), ('listed_output', False), ('log', True), ('maxsum_factor', 1.0), ('mmr_param', 1.0), ('n_keywords', 5), ('openai_ff_max_tokens', 1000), ('outlines_ff_max_tokens', 100), ('rag_chunk_overlap', 128), ('rag_chunk_size', 2048), ('rag_llm', 'llama3.1'), ('reduce_chunk_overlap', 100), ('reduce_chunk_size', 500), ('reduce_max_tokens', 50), ('reduce_temperature', 0.4), ('remove_fields', 'default'), ('retriever_type', 'simple'), ('sampler', 'beam'), ('sampler_beams', 2), ('sampler_temp', 0.4), ('sampler_top_k', 100), ('sampler_top_p', 1), ('similarity_k', 10)]


,hardware_4,organism_part_5,experimental_designs_10,assay_by_molecule_14,study_type_18,experimental_factors_20,total
fullpaper_gpt,6.948866e-01,6.636238e-01,7.430984e-07,8.238029e-01,7.764037e-09,NaN,1.038244e-09
keyword_keyword,2.166848e-08,8.944732e-07,NaN,7.537842e-03,1.758141e-13,0.503445,4.683196e-15
keyword_gpt,NaN,NaN,NaN,5.078125e-01,2.862787e-01,NaN,NaN
keyword_llama,5.240970e-11,9.414096e-04,1.903182e-07,6.906563e-07,4.200394e-07,0.026604,1.075802e-29
keybert_gpt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keybert_llama,4.663011e-11,6.604195e-05,1.903182e-07,6.906563e-07,8.150972e-03,0.041389,1.334012e-26
rag_gpt_17,NaN,8.145294e-01,9.322376e-03,4.302779e-06,2.457328e-04,NaN,2.073052e-07
rag_gpt_10,1.141204e-02,7.186064e-02,3.892417e-03,5.790052e-08,1.781872e-10,NaN,1.772896e-18
rag_llama,4.098215e-12,2.225020e-09,4.320463e-08,8.571764e-11,1.082387e-10,0.016901,1.070352e-44
keybert_keyword,9.236597e-13,1.213155e-07,NaN,6.347656e-03,3.138692e-12,0.167068,5.154234e-23


In [22]:
p(5)

keybert_llama
[('answer_in_quotes', True), ('context_shortener', 'keybert-literal'), ('dataset', 'arxpr2s_25'), ('ff_model', 'llama3.1I-8b-q4'), ('keyphrase_min', 1), ('keyphrase_range_diff', 1), ('listed_output', False), ('log', True), ('maxsum_factor', 1.0), ('mmr_param', 1.0), ('n_keywords', 5), ('openai_ff_max_tokens', 1000), ('outlines_ff_max_tokens', 100), ('rag_chunk_overlap', 128), ('rag_chunk_size', 2048), ('rag_llm', 'llama3.1'), ('reduce_chunk_overlap', 100), ('reduce_chunk_size', 300), ('reduce_max_tokens', 50), ('reduce_temperature', 0.4), ('remove_fields', 'default'), ('retriever_type', 'simple'), ('sampler', 'beam'), ('sampler_beams', 2), ('sampler_temp', 0.4), ('sampler_top_k', 100), ('sampler_top_p', 1), ('similarity_k', 4)]


,hardware_4,organism_part_5,experimental_designs_10,assay_by_molecule_14,study_type_18,experimental_factors_20,total
fullpaper_gpt,NaN,NaN,NaN,NaN,3.114910e-04,NaN,NaN
keyword_keyword,NaN,0.223017,NaN,NaN,2.129056e-09,NaN,NaN
keyword_gpt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keyword_llama,NaN,NaN,NaN,NaN,3.500476e-03,1.0,1.489147e-01
keybert_gpt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keybert_llama,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rag_gpt_17,NaN,NaN,NaN,NaN,1.824224e-01,NaN,NaN
rag_gpt_10,NaN,NaN,NaN,0.757621,3.859616e-06,NaN,NaN
rag_llama,0.375,0.000246,1.0,0.010909,9.633570e-07,0.5,6.231088e-11
keybert_keyword,0.125,0.117185,NaN,NaN,2.906095e-08,NaN,NaN


In [23]:
p(6)

rag_gpt_17
[('answer_in_quotes', True), ('context_shortener', 'rag'), ('dataset', 'arxpr2s_25'), ('ff_model', '4om'), ('keyphrase_min', 1), ('keyphrase_range_diff', 1), ('listed_output', False), ('log', True), ('maxsum_factor', 1.0), ('mmr_param', 1.0), ('n_keywords', 5), ('openai_ff_max_tokens', 1000), ('outlines_ff_max_tokens', 100), ('rag_chunk_overlap', 100), ('rag_chunk_size', 850), ('rag_llm', 'all-minilm:l6-v2'), ('reduce_chunk_overlap', 400), ('reduce_chunk_size', 20000), ('reduce_max_tokens', 50), ('reduce_temperature', 0.4), ('remove_fields', 'default'), ('retriever_type', 'simple'), ('sampler', 'beam'), ('sampler_beams', 2), ('sampler_temp', 0.4), ('sampler_top_k', 100), ('sampler_top_p', 1), ('similarity_k', 17)]


,hardware_4,organism_part_5,experimental_designs_10,assay_by_molecule_14,study_type_18,experimental_factors_20,total
fullpaper_gpt,5.485062e-01,1.000000e+00,0.000729,NaN,2.699796e-03,0.002599,2.980928e-01
keyword_keyword,1.343993e-08,5.744712e-06,NaN,NaN,5.724362e-08,0.001787,4.605776e-04
keyword_gpt,NaN,NaN,NaN,NaN,NaN,0.041389,NaN
keyword_llama,7.749600e-11,7.931924e-03,0.000145,0.434880,1.047575e-01,0.000008,1.179324e-11
keybert_gpt,8.473897e-01,NaN,NaN,NaN,NaN,0.001490,NaN
keybert_llama,2.806114e-11,3.083186e-03,0.000145,0.423340,NaN,0.000013,7.861841e-09
rag_gpt_17,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rag_gpt_10,1.069213e-03,3.906250e-02,0.753906,0.118469,3.855611e-05,1.000000,4.972895e-09
rag_llama,2.463010e-12,1.123642e-09,0.000015,0.000036,2.430496e-05,0.000003,4.902676e-31
keybert_keyword,5.550063e-13,8.235705e-06,NaN,NaN,7.904640e-07,0.000246,7.544171e-07


In [24]:
p(7)

rag_gpt_10
[('answer_in_quotes', True), ('context_shortener', 'rag'), ('dataset', 'arxpr2s_25'), ('ff_model', '4om'), ('keyphrase_min', 1), ('keyphrase_range_diff', 1), ('listed_output', False), ('log', True), ('maxsum_factor', 1.0), ('mmr_param', 1.0), ('n_keywords', 5), ('openai_ff_max_tokens', 1000), ('outlines_ff_max_tokens', 100), ('rag_chunk_overlap', 100), ('rag_chunk_size', 500), ('rag_llm', 'all-minilm:l6-v2'), ('reduce_chunk_overlap', 400), ('reduce_chunk_size', 20000), ('reduce_max_tokens', 50), ('reduce_temperature', 0.4), ('remove_fields', 'default'), ('retriever_type', 'simple'), ('sampler', 'beam'), ('sampler_beams', 2), ('sampler_temp', 0.4), ('sampler_top_k', 100), ('sampler_top_p', 1), ('similarity_k', 10)]


,hardware_4,organism_part_5,experimental_designs_10,assay_by_molecule_14,study_type_18,experimental_factors_20,total
fullpaper_gpt,NaN,NaN,0.002350,NaN,NaN,0.001312,NaN
keyword_keyword,2.479127e-05,1.069213e-03,NaN,NaN,0.039592,0.002700,NaN
keyword_gpt,NaN,NaN,NaN,NaN,NaN,0.063568,NaN
keyword_llama,5.925254e-06,1.138463e-01,0.000519,NaN,NaN,0.000007,8.126397e-03
keybert_gpt,NaN,NaN,NaN,NaN,NaN,0.004344,NaN
keybert_llama,6.906563e-07,1.047575e-01,0.000519,NaN,NaN,0.000012,1.392894e-01
rag_gpt_17,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rag_gpt_10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rag_llama,1.456022e-08,4.066142e-08,0.000061,0.001702,0.492717,0.000002,4.961947e-17
keybert_keyword,8.717444e-09,9.414096e-04,NaN,NaN,0.182422,0.000386,3.110580e-01


In [25]:
p(8)

rag_llama
[('answer_in_quotes', True), ('context_shortener', 'rag'), ('dataset', 'arxpr2s_25'), ('ff_model', 'llama3.1I-8b-q4'), ('keyphrase_min', 1), ('keyphrase_range_diff', 1), ('listed_output', False), ('log', True), ('maxsum_factor', 1.0), ('mmr_param', 1.0), ('n_keywords', 5), ('openai_ff_max_tokens', 1000), ('outlines_ff_max_tokens', 100), ('rag_chunk_overlap', 100), ('rag_chunk_size', 500), ('rag_llm', 'all-minilm:l6-v2'), ('reduce_chunk_overlap', 400), ('reduce_chunk_size', 20000), ('reduce_max_tokens', 50), ('reduce_temperature', 0.4), ('remove_fields', 'default'), ('retriever_type', 'simple'), ('sampler', 'beam'), ('sampler_beams', 2), ('sampler_temp', 0.4), ('sampler_top_k', 100), ('sampler_top_p', 1), ('similarity_k', 5)]


,hardware_4,organism_part_5,experimental_designs_10,assay_by_molecule_14,study_type_18,experimental_factors_20,total
fullpaper_gpt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keyword_keyword,NaN,NaN,NaN,NaN,0.144127,NaN,NaN
keyword_gpt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keyword_llama,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keybert_gpt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keybert_llama,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rag_gpt_17,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rag_gpt_10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
rag_llama,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keybert_keyword,1.0,NaN,NaN,NaN,0.516412,NaN,NaN


In [26]:
p(9)

keybert_keyword
[('answer_in_quotes', True), ('context_shortener', 'keybert-literal'), ('dataset', 'arxpr2s_25'), ('ff_model', 'keybert'), ('keyphrase_min', 1), ('keyphrase_range_diff', 1), ('listed_output', False), ('log', True), ('maxsum_factor', 1.0), ('mmr_param', 1.0), ('n_keywords', 5), ('openai_ff_max_tokens', 1000), ('outlines_ff_max_tokens', 100), ('rag_chunk_overlap', 128), ('rag_chunk_size', 2048), ('rag_llm', 'llama3.1'), ('reduce_chunk_overlap', 100), ('reduce_chunk_size', 500), ('reduce_max_tokens', 50), ('reduce_temperature', 0.4), ('remove_fields', 'default'), ('retriever_type', 'simple'), ('sampler', 'beam'), ('sampler_beams', 2), ('sampler_temp', 0.4), ('sampler_top_k', 100), ('sampler_top_p', 1), ('similarity_k', 1)]


,hardware_4,organism_part_5,experimental_designs_10,assay_by_molecule_14,study_type_18,experimental_factors_20,total
fullpaper_gpt,NaN,NaN,7.540127e-09,NaN,NaN,NaN,NaN
keyword_keyword,NaN,NaN,NaN,8.318119e-01,0.42395,NaN,NaN
keyword_gpt,NaN,NaN,1.495414e-01,NaN,NaN,NaN,NaN
keyword_llama,NaN,NaN,1.874468e-09,1.039363e-03,NaN,0.423950,9.896015e-02
keybert_gpt,NaN,NaN,1.171851e-01,NaN,NaN,NaN,NaN
keybert_llama,NaN,NaN,1.874468e-09,7.718664e-04,NaN,0.607239,6.996756e-01
rag_gpt_17,NaN,NaN,5.042182e-04,9.444166e-03,NaN,NaN,NaN
rag_gpt_10,NaN,NaN,2.967323e-04,3.281542e-04,NaN,NaN,NaN
rag_llama,NaN,0.027992,4.238055e-10,1.450309e-07,NaN,0.301758,7.068222e-11
keybert_keyword,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
p(10)

keyword_mistral
[('answer_in_quotes', True), ('context_shortener', 'keyword-literal'), ('dataset', 'arxpr2s_25'), ('ff_model', 'TheBloke/Mistral-7B-v0.1-GPTQ'), ('keyphrase_min', 1), ('keyphrase_range_diff', 1), ('listed_output', False), ('log', True), ('maxsum_factor', 1.0), ('mmr_param', 1.0), ('n_keywords', 5), ('openai_ff_max_tokens', 1000), ('outlines_ff_max_tokens', 100), ('rag_chunk_overlap', 128), ('rag_chunk_size', 2048), ('rag_llm', 'llama3.1'), ('reduce_chunk_overlap', 100), ('reduce_chunk_size', 300), ('reduce_max_tokens', 50), ('reduce_temperature', 0.4), ('remove_fields', 'default'), ('retriever_type', 'simple'), ('sampler', 'beam'), ('sampler_beams', 2), ('sampler_temp', 0.4), ('sampler_top_k', 100), ('sampler_top_p', 1), ('similarity_k', 4)]


,hardware_4,organism_part_5,experimental_designs_10,assay_by_molecule_14,study_type_18,experimental_factors_20,total
fullpaper_gpt,NaN,NaN,1.000,NaN,NaN,NaN,NaN
keyword_keyword,2.305526e-04,0.096252,NaN,4.986020e-02,0.055511,0.694887,NaN
keyword_gpt,NaN,NaN,NaN,1.000000e+00,NaN,NaN,NaN
keyword_llama,7.117887e-06,1.000000,0.625,9.758408e-06,NaN,0.078354,6.908518e-04
keybert_gpt,NaN,NaN,NaN,NaN,NaN,NaN,NaN
keybert_llama,2.600383e-06,0.852684,0.625,9.758408e-06,NaN,0.115318,4.532756e-02
rag_gpt_17,NaN,NaN,NaN,1.478023e-04,NaN,NaN,NaN
rag_gpt_10,NaN,NaN,NaN,6.131171e-06,NaN,NaN,7.662240e-01
rag_llama,4.320463e-08,0.000231,0.250,3.191281e-09,0.621873,0.041389,1.183314e-15
keybert_keyword,2.580284e-08,0.063915,NaN,1.153183e-01,0.285751,0.317311,1.306461e-01


In [30]:
p(11)

keylist_gpt
[('answer_in_quotes', True), ('context_shortener', 'keyword-list'), ('dataset', 'arxpr2s_25'), ('ff_model', '4om'), ('keyphrase_min', 1), ('keyphrase_range_diff', 1), ('listed_output', False), ('log', True), ('maxsum_factor', 1.0), ('mmr_param', 1.0), ('n_keywords', 5), ('openai_ff_max_tokens', 1000), ('outlines_ff_max_tokens', 100), ('rag_chunk_overlap', 100), ('rag_chunk_size', 500), ('rag_llm', 'llama3.1'), ('reduce_chunk_overlap', 400), ('reduce_chunk_size', 20000), ('reduce_max_tokens', 50), ('reduce_temperature', 0.4), ('remove_fields', 'default'), ('retriever_type', 'simple'), ('sampler', 'beam'), ('sampler_beams', 2), ('sampler_temp', 0.4), ('sampler_top_k', 100), ('sampler_top_p', 1), ('similarity_k', 10)]


,hardware_4,organism_part_5,experimental_designs_10,assay_by_molecule_14,study_type_18,experimental_factors_20,total
fullpaper_gpt,NaN,6.290588e-01,1.238710e-06,NaN,8.683228e-07,0.000855,2.283480e-07
keyword_keyword,9.030489e-08,2.065286e-06,NaN,1.338005e-01,6.582202e-11,0.000858,1.211767e-11
keyword_gpt,NaN,NaN,NaN,NaN,NaN,0.026604,NaN
keyword_llama,5.919399e-10,3.004262e-03,3.186355e-07,9.047332e-06,9.448129e-05,0.000009,1.176331e-23
keybert_gpt,NaN,NaN,NaN,NaN,NaN,0.001544,NaN
keybert_llama,4.891710e-10,2.556309e-04,3.186355e-07,1.536009e-05,1.779317e-01,0.000015,2.028075e-21
rag_gpt_17,NaN,7.744141e-01,1.181793e-02,1.599247e-05,5.345677e-03,1.000000,8.708276e-06
rag_gpt_10,3.258280e-02,7.835388e-02,6.610751e-03,1.903182e-07,1.279841e-08,0.790527,3.860327e-16
rag_llama,1.889897e-11,4.040854e-10,7.237830e-08,2.952349e-10,6.706293e-09,0.000003,3.672372e-42
keybert_keyword,4.262192e-12,3.061253e-06,NaN,2.862787e-01,1.075446e-10,0.000156,4.200869e-17


In [31]:
p(12)

keydescript_gpt
[('answer_in_quotes', True), ('context_shortener', 'keyword-fielddescription'), ('dataset', 'arxpr2s_25'), ('ff_model', '4om'), ('keyphrase_min', 1), ('keyphrase_range_diff', 1), ('listed_output', False), ('log', True), ('maxsum_factor', 1.0), ('mmr_param', 1.0), ('n_keywords', 5), ('openai_ff_max_tokens', 1000), ('outlines_ff_max_tokens', 100), ('rag_chunk_overlap', 100), ('rag_chunk_size', 500), ('rag_llm', 'llama3.1'), ('reduce_chunk_overlap', 400), ('reduce_chunk_size', 20000), ('reduce_max_tokens', 50), ('reduce_temperature', 0.4), ('remove_fields', 'default'), ('retriever_type', 'simple'), ('sampler', 'beam'), ('sampler_beams', 2), ('sampler_temp', 0.4), ('sampler_top_k', 100), ('sampler_top_p', 1), ('similarity_k', 10)]


,hardware_4,organism_part_5,experimental_designs_10,assay_by_molecule_14,study_type_18,experimental_factors_20,total
fullpaper_gpt,8.318119e-01,2.127075e-02,3.444129e-06,NaN,1.613164e-04,0.002577,1.529046e-06
keyword_keyword,2.166848e-08,2.432744e-08,NaN,5.412562e-01,1.541726e-08,0.002022,4.951565e-10
keyword_gpt,NaN,2.378845e-01,NaN,NaN,NaN,0.078354,NaN
keyword_llama,5.240970e-11,1.468685e-05,8.944732e-07,1.560888e-04,3.480848e-02,0.000040,8.115702e-22
keybert_gpt,NaN,1.670685e-01,NaN,NaN,NaN,0.004344,NaN
keybert_llama,4.663011e-11,2.065286e-06,8.944732e-07,3.281542e-04,NaN,0.000063,9.582115e-18
rag_gpt_17,NaN,4.904175e-02,2.127075e-02,2.771616e-04,5.234671e-01,NaN,8.161930e-05
rag_gpt_10,4.677735e-03,1.543880e-03,1.181793e-02,1.499876e-06,1.571198e-06,NaN,1.158781e-16
rag_llama,4.098215e-12,1.135214e-11,2.034555e-07,2.225020e-09,6.025761e-06,0.000015,1.292826e-39
keybert_keyword,9.236597e-13,4.600924e-08,NaN,7.054570e-01,1.108891e-07,0.000386,1.044351e-14


In [27]:
scores = []
for i, key in enumerate(list(data.keys())):
    scores.append([])
    for field in data[key]:
        scores[i].append(sum(data[key][field])/len(data[key][field]))
scores

df = pd.DataFrame(data=scores, columns= data[key].keys(), index=shortnames).sort_values("total", ascending=False)
df = df.drop("experimental_factors_20", axis = 1)
print(df.to_markdown())
print("")


|                 |   hardware_4 |   organism_part_5 |   experimental_designs_10 |   assay_by_molecule_14 |   study_type_18 |   total |
|:----------------|-------------:|------------------:|--------------------------:|-----------------------:|----------------:|--------:|
| keyword_gpt     |         0.64 |              0.71 |                      0.3  |                   0.84 |            0.69 |   0.636 |
| keybert_gpt     |         0.52 |              0.7  |                      0.3  |                   0.87 |            0.75 |   0.628 |
| keylist_gpt     |         0.49 |              0.7  |                      0.29 |                   0.83 |            0.68 |   0.598 |
| keydescript_gpt |         0.52 |              0.77 |                      0.27 |                   0.79 |            0.57 |   0.584 |
| rag_gpt_17      |         0.53 |              0.68 |                      0.17 |                   0.61 |            0.53 |   0.504 |
| fullpaper_gpt   |         0.5  |              

In [28]:
lt = df.to_latex()
lt = lt.replace("000 ", " ")
lt = lt.replace("_", " ")
print(lt)

\begin{tabular}{lrrrrrr}
\toprule
 & hardware 4 & organism part 5 & experimental designs 10 & assay by molecule 14 & study type 18 & total \\
\midrule
keyword gpt & 0.640 & 0.710 & 0.300 & 0.840 & 0.690 & 0.636 \\
keybert gpt & 0.520 & 0.700 & 0.300 & 0.870 & 0.750 & 0.628 \\
keylist gpt & 0.490 & 0.700 & 0.290 & 0.830 & 0.680 & 0.598 \\
keydescript gpt & 0.520 & 0.770 & 0.270 & 0.790 & 0.570 & 0.584 \\
rag gpt 17 & 0.530 & 0.680 & 0.170 & 0.610 & 0.530 & 0.504 \\
fullpaper gpt & 0.500 & 0.670 & 0.020 & 0.850 & 0.350 & 0.478 \\
keyword keyword & 0.090 & 0.440 & 0.550 & 0.750 & 0.170 & 0.400 \\
keyword mistral & 0.320 & 0.520 & 0.030 & 0.850 & 0.280 & 0.400 \\
rag gpt 10 & 0.360 & 0.610 & 0.150 & 0.540 & 0.290 & 0.390 \\
keybert keyword & 0.010 & 0.420 & 0.390 & 0.770 & 0.210 & 0.360 \\
keybert llama & 0.050 & 0.510 & 0.010 & 0.560 & 0.610 & 0.348 \\
keyword llama & 0.070 & 0.510 & 0.010 & 0.560 & 0.430 & 0.316 \\
rag llama & 0.020 & 0.290 & 0.000 & 0.380 & 0.250 & 0.188 \\
\bottomrule
